# Gaussian Naive Bayes 2 - From Scratch, Object-Oriented

> **MLCourse · Machine Learning · 04_naive_bayes**

We implement `MyGaussianNB` with log-space likelihoods, per-class priors, and `predict_proba` - then verify it against scikit-learn on REAL Titanic data.

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# load real data - Titanic
titanic = sns.load_dataset('titanic')
# Keep relevant features, drop missing
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']].dropna()
titanic['sex'] = (titanic['sex'] == 'male').astype(int)
X = titanic.drop(columns='survived').values
y = titanic['survived'].values

print('Data shape:', X.shape, 'Class balance:', np.bincount(y))

Data shape: (714, 6) Class balance: [424 290]


### 1. Designing the class

Public API (mirrors sklearn so it is swappable):

| method | job |
|---|---|
| `fit(X, y)` | compute class priors, per-class feature means & variances |
| `predict(X)` | argmax of log-posterior |
| `predict_proba(X)` | normalized posterior probabilities |
| `score(X, y)` | return accuracy |

Key insight: **log-space computation** prevents underflow when multiplying many small probabilities.

Math: P(y|x) ∝ P(y) ∏ P(x_j|y) → log P(y|x) = log P(y) + ∑ log P(x_j|y)

In [2]:
class MyGaussianNB:
    """
    Gaussian Naive Bayes classifier from scratch.

    Parameters
    ----------
    var_smoothing : float, default=1e-9
        Portion of the largest variance to add to all variances for numerical stability.

    Attributes after fit
    --------------------
    class_prior_ : ndarray (n_classes,) - P(y)
    theta_ : ndarray (n_classes, n_features) - mean of each feature per class
    var_ : ndarray (n_classes, n_features) - variance of each feature per class
    epsilon_ : float - smoothing applied to variances
    classes_ : ndarray - unique class labels
    """

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    # fit
    def fit(self, X, y):
        """Estimate the class priors and class-wise feature Gaussians (MLE)."""
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int).ravel()

        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        self.theta_ = np.zeros((n_classes, n_features))
        self.var_ = np.zeros((n_classes, n_features))
        self.class_prior_ = np.zeros(n_classes)

        for i, cls in enumerate(self.classes_):
            X_cls = X[y == cls]
            self.theta_[i] = X_cls.mean(axis=0)            # mu_j for class i
            self.var_[i] = X_cls.var(axis=0, ddof=0)       # sigma_j^2 (MLE)
            self.class_prior_[i] = len(X_cls) / len(X)     # P(y=i) = n_i / n

        # Add smoothing to variances (prevents zero variance)
        self.epsilon_ = self.var_smoothing * self.var_.max()
        self.var_ += self.epsilon_

        return self

    # log likelihood
    def _log_likelihood(self, X):
        """Compute log P(x|y) for each class - shape (n_samples, n_classes).

        Gaussian PDF:  P(x_j | y) = (1/sqrt(2*pi*sigma^2)) * exp( -(x_j - mu)^2/(2*sigma^2) )
        log P(x | y) = sum_j [ -0.5*log(2*pi*sigma_j^2) - (x_j - mu_j)^2/(2*sigma_j^2) ]
        """
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        log_likelihood = np.zeros((n_samples, n_classes))

        for i in range(n_classes):
            diff = X - self.theta_[i]                       # x - mu
            # -0.5 * sum_j [ log(2*pi*sigma^2) + (x-mu)^2/sigma^2 ]
            log_likelihood[:, i] = -0.5 * np.sum(
                np.log(2 * np.pi * self.var_[i]) + (diff ** 2) / self.var_[i],
                axis=1
            )
        return log_likelihood

    # joint log likelihood
    def _joint_log_likelihood(self, X):
        """Compute log P(y) + log P(x|y) = log P(y, x)   (naive-Bayes product)."""
        return self._log_likelihood(X) + np.log(self.class_prior_)

    # predict_proba
    def predict_proba(self, X):
        """Return posterior probabilities P(y|x) for each class.

        Bayes rule:  P(y|x) = P(y)P(x|y) / P(x)
        Computed in log space, then normalized via log-sum-exp trick:
            P(y|x) = exp(jll - max_jll) / sum_c exp(jll_c - max_jll)
        """
        X = np.asarray(X, dtype=float)
        jll = self._joint_log_likelihood(X)                 # log P(y, x)
        max_jll = np.max(jll, axis=1, keepdims=True)
        exp_jll = np.exp(jll - max_jll)                     # stability: shift by max
        return exp_jll / exp_jll.sum(axis=1, keepdims=True) # normalise to a distribution

    # predict
    def predict(self, X):
        """Predict class = argmax_c log P(y=c, x)."""
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]

    # score
    def score(self, X, y):
        """Return prediction accuracy on (X, y)."""
        return accuracy_score(y, self.predict(X))

    def __repr__(self):
        return f"MyGaussianNB(var_smoothing={self.var_smoothing})"

print('MyGaussianNB class defined')


MyGaussianNB class defined


### 2. Real data prep - Titanic survival

Naive Bayes is scale-invariant for GaussianNB, so no scaler needed.

In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Our implementation
mine = MyGaussianNB().fit(X_tr, y_tr)
pred_mine = mine.predict(X_te)
acc_mine = accuracy_score(y_te, pred_mine)

# sklearn
sk = GaussianNB().fit(X_tr, y_tr)
pred_sk = sk.predict(X_te)
acc_sk = accuracy_score(y_te, pred_sk)

print(f'MyGaussianNB accuracy:  {acc_mine:.4f}')
print(f'sklearn GaussianNB accuracy: {acc_sk:.4f}')
print(f'Difference: {abs(acc_mine - acc_sk):.4f}')

MyGaussianNB accuracy:  0.7654
sklearn GaussianNB accuracy: 0.7654
Difference: 0.0000


### 3. Coefficient/parameter comparison

Compare learned means, variances, and priors.

In [4]:
feature_names = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
class_names = ['Died (0)', 'Survived (1)']

print('=== Class Priors ===')
for i, cls in enumerate(class_names):
    print(f'  {cls}: mine={mine.class_prior_[i]:.4f}, sklearn={sk.class_prior_[i]:.4f}')

print('\n=== Feature Means (theta_) ===')
df_theta = pd.DataFrame(mine.theta_, index=class_names, columns=feature_names)
print(df_theta.round(3))

print('\n=== Feature Variances (var_) ===')
df_var = pd.DataFrame(mine.var_, index=class_names, columns=feature_names)
print(df_var.round(3))

=== Class Priors ===
  Died (0): mine=0.5944, sklearn=0.5944
  Survived (1): mine=0.4056, sklearn=0.4056

=== Feature Means (theta_) ===
              pclass    sex     age  sibsp  parch    fare
Died (0)       2.487  0.852  30.561  0.509  0.374  22.844
Survived (1)   1.862  0.327  27.808  0.521  0.507  50.862

=== Feature Variances (var_) ===
              pclass    sex      age  sibsp  parch      fare
Died (0)       0.564  0.126  205.865  1.137  0.825  1124.783
Survived (1)   0.681  0.220  220.301  0.572  0.545  4815.871


### 4. Probability calibration check

Verify predict_proba outputs valid, well-calibrated probabilities.

In [5]:
probas_mine = mine.predict_proba(X_te)
probas_sk = sk.predict_proba(X_te)

print('Probability sums (should be 1.0):', np.unique(np.round(probas_mine.sum(axis=1), 10)))
print('First 10 predictions (mine vs sklearn):')
for i in range(min(10, len(X_te))):
    print(f'  Sample {i}: mine={probas_mine[i]}, sklearn={probas_sk[i]}, diff={np.abs(probas_mine[i]-probas_sk[i]).max():.6f}')

Probability sums (should be 1.0): [1.]
First 10 predictions (mine vs sklearn):
  Sample 0: mine=[9.99983066e-01 1.69340216e-05], sklearn=[9.99983067e-01 1.69329787e-05], diff=0.000000
  Sample 1: mine=[0.85130709 0.14869291], sklearn=[0.85130839 0.14869161], diff=0.000001
  Sample 2: mine=[0.91935774 0.08064226], sklearn=[0.91935873 0.08064127], diff=0.000001
  Sample 3: mine=[0.9520497 0.0479503], sklearn=[0.95205032 0.04794968], diff=0.000001
  Sample 4: mine=[0.95208423 0.04791577], sklearn=[0.95208484 0.04791516], diff=0.000001
  Sample 5: mine=[0.3227832 0.6772168], sklearn=[0.32278452 0.67721548], diff=0.000001
  Sample 6: mine=[0.91551752 0.08448248], sklearn=[0.91551856 0.08448144], diff=0.000001
  Sample 7: mine=[0.00954151 0.99045849], sklearn=[0.00954108 0.99045892], diff=0.000000
  Sample 8: mine=[0.40356894 0.59643106], sklearn=[0.40357041 0.59642959], diff=0.000001
  Sample 9: mine=[0.32419164 0.67580836], sklearn=[0.32418315 0.67581685], diff=0.000008


### 5. Log-likelihood contribution per feature

Show how each feature contributes to the decision for a few test samples.

In [6]:
# Show log-likelihood breakdown for first 3 test samples
X_sample = X_te[:3]
log_lik = mine._log_likelihood(X_sample)
log_prior = np.log(mine.class_prior_)
joint = log_lik + log_prior

for i in range(3):
    print(f'\nSample {i} (true label: {y_te[i]}):')
    for c_idx, cls in enumerate(class_names):
        print(f'  {cls}:')
        print(f'    log P(y) = {log_prior[c_idx]:.3f}')
        for f_idx, feat in enumerate(feature_names):
            ll = log_lik[i, c_idx]
            # Per-feature contribution (approximate)
            diff = X_sample[i, f_idx] - mine.theta_[c_idx, f_idx]
            var = mine.var_[c_idx, f_idx]
            feat_ll = -0.5 * (np.log(2 * np.pi * var) + diff**2 / var)
            print(f'    {feat}: {feat_ll:.3f}')
        print(f'    Total log P(x|y) = {log_lik[i, c_idx]:.3f}')
        print(f'    Joint log P(y,x) = {joint[i, c_idx]:.3f}')
    pred = mine.classes_[np.argmax(joint[i])]
    print(f'  → Prediction: {pred}')


Sample 0 (true label: 0):
  Died (0):
    log P(y) = -0.520
    pclass: -0.866
    sex: 0.030
    age: -5.705
    sibsp: -9.853
    parch: -2.424
    fare: -4.689
    Total log P(x|y) = -23.506
    Joint log P(y,x) = -24.027
  Survived (1):
    log P(y) = -0.902
    pclass: -1.678
    sex: -1.190
    age: -5.248
    sibsp: -18.173
    parch: -2.661
    fare: -5.160
    Total log P(x|y) = -34.110
    Joint log P(y,x) = -35.013
  → Prediction: 0

Sample 1 (true label: 0):
  Died (0):
    log P(y) = -0.520
    pclass: -0.843
    sex: 0.030
    age: -3.583
    sibsp: -1.089
    parch: -1.060
    fare: -4.521
    Total log P(x|y) = -11.066
    Joint log P(y,x) = -11.586
  Survived (1):
    log P(y) = -0.902
    pclass: -0.741
    sex: -1.190
    age: -3.640
    sibsp: -0.840
    parch: -0.838
    fare: -5.179
    Total log P(x|y) = -12.429
    Joint log P(y,x) = -13.331
  → Prediction: 0

Sample 2 (true label: 1):
  Died (0):
    log P(y) = -0.520
    pclass: -0.866
    sex: 0.030
    age:

### Summary & key takeaways

- Naive Bayes assumes **conditional independence**: P(x|y) = ∏ P(x_j|y). Usually wrong but ranking often correct.
- **Log-space computation** prevents underflow: log P(y|x) = log P(y) + ∑ log P(x_j|y).
- GaussianNB: P(x_j|y) = N(μ, σ²) per feature per class - only needs mean & variance.
- **No scaling needed** for GaussianNB - it's naturally scale-invariant.
- `var_smoothing` prevents zero variance (numerical stability) - rarely needs tuning.
- Probabilities are often over-confident but class rankings are reliable.
- Our pure-NumPy implementation matches sklearn's GaussianNB on real data perfectly.